#### PATTERN PROFILE

##### 05.1 DOCUMENTAÇÃO DO PATTERN PROFILE

###### Objetivo

O *Pattern Profile* tem como objetivo identificar padrões estruturais presentes nos valores das colunas do DataFrame.

Enquanto o *Schema Profile* analisa a estrutura das colunas e o *Distribution Profile* analisa a distribuição dos valores, o *Pattern Profile* analisa a forma como os valores estão estruturados.

A análise não depende do significado de negócio das colunas.

O foco está nas características observáveis dos valores, como:

- comprimento;
- composição de caracteres;
- estrutura;
- padrões predominantes;
- diversidade estrutural;
- concentração estrutural;
- regularidade;
- desvios em relação ao padrão predominante.

###### Exemplos

Valores:

`ABC123`

`DEF456`

`XYZ789`

apresentam o padrão estrutural:

`LLLNNN`

Enquanto valores como:

`12/05/2026`

`31/08/2026`

apresentam:

`NNXNNXNNNN`

###### Principais análises

- Comprimento dos valores;
- Composição de caracteres;
- Estruturas predominantes;
- Frequência dos padrões;
- Concentração estrutural;
- Diversidade estrutural;
- Regularidade estrutural;
- Desvios em relação ao padrão predominante;
- Perfil consolidado dos padrões.

###### Característica

O *Pattern Profile* é independente da origem dos dados.

Sua análise é realizada exclusivamente sobre o DataFrame disponibilizado pelo *Data Preparation*.

Dessa forma, o mesmo Profile pode ser utilizado em diferentes fontes e estruturas de dados.

###### Resultado esperado

Ao final desta etapa estarão disponíveis:

- padrões predominantes por coluna;
- percentual de predominância;
- quantidade de estruturas distintas;
- concentração Top 1, Top 3 e Top 5;
- diversidade estrutural;
- regularidade estrutural;
- percentual de desvio estrutural;
- identificação de possíveis outliers de padrão;
- DataFrame consolidado do Pattern Profile.

In [0]:
%run "./02_DATA_PREPARATION"

In [0]:
# ============================================================
# 05.3 PARÂMETROS DO PROFILE
# ============================================================

# O QUE FAZ:
# Define os parâmetros específicos utilizados pelo Pattern Profile.

# COMO FAZ:
# Define a quantidade de padrões analisados nas métricas Top N e os níveis utilizados para concentração estrutural.

# POR QUE É IMPORTANTE:
# Mantém as configurações específicas do Profile dentro do próprio notebook, sem sobrecarregar a biblioteca central.

# PERGUNTA RESPONDIDA:
# "Quais parâmetros serão utilizados na análise de padrões?"

PATTERN_TOP_N = 5

PATTERN_CONCENTRATION_N = [1, 3, 5]

print(f"PATTERN_TOP_N: {PATTERN_TOP_N}")
print(
    f"PATTERN_CONCENTRATION_N: {PATTERN_CONCENTRATION_N}"
)

In [0]:
# ============================================================
# 05.4 VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame preparado e as informações necessárias para o Pattern Profile estão disponíveis.

# COMO FAZ:
# Verifica o DataFrame preparado, o total de registros e as colunas elegíveis para análise de padrões.

# POR QUE É IMPORTANTE:
# Evita a execução do Profile sobre uma estrutura inválida.

# PERGUNTA RESPONDIDA:
# "O DataFrame possui as informações necessárias para executar o Pattern Profile?"

if "df_prepared" not in locals():
    raise ValueError(
        "O DataFrame 'df_prepared' não foi disponibilizado."
    )

if "total_registros" not in locals():
    raise ValueError(
        "A variável 'total_registros' não foi disponibilizada."
    )

if "colunas_pattern" not in locals():
    raise ValueError(
        "A variável 'colunas_pattern' não foi disponibilizada."
    )

if total_registros == 0:
    raise ValueError(
        "O DataFrame preparado não possui registros."
    )

if len(colunas_pattern) == 0:
    raise ValueError(
        "Não existem colunas elegíveis para o Pattern Profile."
    )

print("DataFrame validado com sucesso.")
print(f"Total de registros: {total_registros}")
print(f"Colunas analisáveis: {len(colunas_pattern)}")

In [0]:
# ============================================================
# 05.5 RESUMO EXECUTIVO DOS PADRÕES POR COLUNA
# ============================================================

# O QUE FAZ:
# Identifica os padrões estruturais encontrados em cada coluna e sua frequência de ocorrência.

# COMO FAZ:
# Utiliza a função reutilizável do notebook de biblioteca.

# POR QUE É IMPORTANTE:
# Cria a base principal para as análises posteriores de concentração, diversidade, regularidade e desvios.

# PERGUNTA RESPONDIDA:
# "Quais padrões estruturais existem em cada coluna?"

pattern_summary_df = generate_pattern_summary(
    df_prepared,
    colunas_pattern,
    total_registros
)

display(
    pattern_summary_df
    .orderBy(
        "coluna",
        "ranking"
    )
)

In [0]:
# ============================================================
# 05.6 COMPRIMENTO DOS VALORES
# ============================================================

# O QUE FAZ:
# Analisa a distribuição do comprimento dos valores em cada coluna.

# COMO FAZ:
# Calcula o tamanho dos valores e agrupa os registros pela quantidade de caracteres.

# POR QUE É IMPORTANTE:
# Alterações no comprimento podem indicar inconsistências, formatos inesperados ou mudanças no padrão de origem.

# PERGUNTA RESPONDIDA:
# "Quais comprimentos de valor são predominantes em cada coluna?"

comprimento_resultados = []

for coluna in colunas_pattern:

    comprimento_coluna = (
        df_prepared
        .select(
            F.lit(coluna).alias("coluna"),
            add_pattern_length(
                F.col(coluna)
            ).alias("comprimento")
        )
        .groupBy(
            "coluna",
            "comprimento"
        )
        .agg(
            F.count("*").alias("frequencia")
        )
        .withColumn(
            "pct_comprimento",
            F.col("frequencia")
            / F.lit(total_registros)
            * 100
        )
    )

    comprimento_resultados.append(
        comprimento_coluna
    )

pattern_length_df = comprimento_resultados[0]

for resultado in comprimento_resultados[1:]:
    pattern_length_df = pattern_length_df.unionByName(
        resultado
    )

display(
    pattern_length_df.orderBy(
        "coluna",
        F.desc("frequencia")
    )
)

In [0]:
# ============================================================
# 05.7 COMPOSIÇÃO DE CARACTERES
# ============================================================

# O QUE FAZ:
# Identifica a composição predominante dos valores de cada coluna.

# COMO FAZ:
# Utiliza as categorias:
#
# NUMÉRICO
# ALFABÉTICO
# ALFANUMÉRICO
# SÍMBOLOS
# MISTO
# VAZIO

# POR QUE É IMPORTANTE:
# Permite identificar alterações no tipo estrutural dos valores, mesmo quando o tipo Spark da coluna permanece inalterado.

# PERGUNTA RESPONDIDA:
# "Os valores são predominantemente numéricos, textuais, alfanuméricos ou mistos?"

character_composition_df = generate_character_composition(
    df_prepared,
    colunas_pattern,
    total_registros
)

display(
    character_composition_df.orderBy(
        "coluna",
        F.desc("frequencia")
    )
)

In [0]:
# ============================================================
# 05.8 ESTRUTURAS PREDOMINANTES
# ============================================================

# O QUE FAZ:
# Seleciona os padrões estruturais mais frequentes de cada coluna.

# COMO FAZ:
# Utiliza o ranking produzido pelo Pattern Summary.

# POR QUE É IMPORTANTE:
# Permite identificar quais estruturas dominam cada atributo.

# PERGUNTA RESPONDIDA:
# "Quais são as principais estruturas presentes em cada coluna?"

pattern_top_df = (
    pattern_summary_df
    .filter(
        F.col("ranking") <= PATTERN_TOP_N
    )
    .select(
        "coluna",
        "estrutura",
        "frequencia",
        "pct_estrutura",
        "ranking"
    )
    .orderBy(
        "coluna",
        "ranking"
    )
)

display(pattern_top_df)

In [0]:
# ============================================================
# 05.9 CONCENTRAÇÃO ESTRUTURAL
# ============================================================

# O QUE FAZ:
# Calcula quanto da coluna está concentrado nas estruturas mais frequentes.

# COMO FAZ:
# Soma o percentual das estruturas que pertencem aos Top 1, Top 3 e Top 5 padrões.

# POR QUE É IMPORTANTE:
# Permite identificar colunas altamente padronizadas ou com grande variedade estrutural.

# PERGUNTA RESPONDIDA:
# "Quanto da coluna está concentrado nos padrões predominantes?"

pattern_concentration_df = (
    pattern_summary_df
    .groupBy("coluna")
    .agg(
        *[
            F.sum(
                F.when(
                    F.col("ranking") <= n,
                    F.col("pct_estrutura")
                ).otherwise(0)
            ).alias(
                f"concentracao_top{n}"
            )
            for n in PATTERN_CONCENTRATION_N
        ]
    )
)

display(
    pattern_concentration_df.orderBy(
        F.desc("concentracao_top1")
    )
)

In [0]:
# ============================================================
# 05.10 DIVERSIDADE ESTRUTURAL
# ============================================================

# O QUE FAZ:
# Calcula a quantidade de estruturas distintas observadas em cada coluna.

# COMO FAZ:
# Conta a quantidade de estruturas diferentes identificadas pelo Pattern Summary.

# POR QUE É IMPORTANTE:
# Uma grande quantidade de estruturas distintas pode indicar baixa padronização dos dados.

# PERGUNTA RESPONDIDA:
# "Quantas estruturas diferentes existem em cada coluna?"

pattern_diversity_df = (
    pattern_summary_df
    .groupBy("coluna")
    .agg(
        F.countDistinct(
            "estrutura"
        ).alias(
            "estruturas_distintas"
        )
    )
)

display(
    pattern_diversity_df.orderBy(
        F.desc("estruturas_distintas")
    )
)

In [0]:
# ============================================================
# 05.11 REGULARIDADE ESTRUTURAL
# ============================================================

# O QUE FAZ:
# Classifica a regularidade estrutural de cada coluna.

# COMO FAZ:
# Utiliza a concentração Top 1 para identificar o grau de predominância de uma única estrutura.

# POR QUE É IMPORTANTE:
# Uma estrutura altamente predominante indica maior regularidade estrutural.

# PERGUNTA RESPONDIDA:
# "A coluna apresenta uma estrutura predominantemente regular?"

pattern_regularity_df = (
    pattern_concentration_df
    .withColumn(
        "classificacao_regularidade",
        F.when(
            F.col("concentracao_top1") >= 95,
            "ALTAMENTE REGULAR"
        )
        .when(
            F.col("concentracao_top1") >= 80,
            "REGULAR"
        )
        .when(
            F.col("concentracao_top1") >= 50,
            "MODERADAMENTE REGULAR"
        )
        .otherwise(
            "IRREGULAR"
        )
    )
)

display(
    pattern_regularity_df.orderBy(
        F.desc("concentracao_top1")
    )
)

In [0]:
# ============================================================
# 05.12 DETECÇÃO DE OUTLIERS DE PADRÃO
# ============================================================

# O QUE FAZ:
# Identifica registros cujo padrão estrutural é diferente do padrão predominante da respectiva coluna.

# COMO FAZ:
# Primeiro identifica o padrão de maior frequência em cada coluna. Em seguida, compara cada registro com esse padrão.

# POR QUE É IMPORTANTE:
# Permite identificar valores estruturalmente diferentes daqueles normalmente observados.

# PERGUNTA RESPONDIDA:
# "Quais registros apresentam uma estrutura diferente do padrão predominante?"

pattern_dominante_df = (
    pattern_summary_df
    .filter(
        F.col("ranking") == 1
    )
    .select(
        "coluna",
        F.col("estrutura").alias(
            "estrutura_dominante"
        )
    )
)

outlier_resultados = []

for coluna in colunas_pattern:

    estrutura_atual = generate_pattern_structure(
        F.col(coluna)
    )

    dominante = (
        pattern_dominante_df
        .filter(
            F.col("coluna") == coluna
        )
        .select(
            "estrutura_dominante"
        )
        .first()
    )

    if dominante:

        estrutura_dominante = dominante[
            "estrutura_dominante"
        ]

        resultado = (
            df_prepared
            .select(
                F.lit(coluna).alias("coluna"),
                estrutura_atual.alias(
                    "estrutura_observada"
                )
            )
            .withColumn(
                "estrutura_dominante",
                F.lit(estrutura_dominante)
            )
            .withColumn(
                "eh_desvio",
                F.when(
                    F.col("estrutura_observada") !=
                    F.col("estrutura_dominante"),
                    1
                ).otherwise(0)
            )
        )

        outlier_resultados.append(
            resultado
        )

pattern_outlier_df = outlier_resultados[0]

for resultado in outlier_resultados[1:]:
    pattern_outlier_df = pattern_outlier_df.unionByName(
        resultado
    )

display(
    pattern_outlier_df
    .filter(
        F.col("eh_desvio") == 1
    )
)

In [0]:
# ============================================================
# 05.13 PERFIL CONSOLIDADO DE PADRÕES
# ============================================================

# O QUE FAZ:
# Consolida as principais métricas produzidas pelo Pattern Profile.

# COMO FAZ:
# Realiza joins entre:
#
# Pattern Summary
# Pattern Concentration
# Pattern Diversity
# Pattern Regularity
#
# e calcula o percentual de desvio estrutural.

# POR QUE É IMPORTANTE:
# Cria uma única estrutura de resultados para utilização posterior pelo Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Qual é o perfil estrutural completo de cada coluna?"

pattern_deviation_summary_df = (
    pattern_summary_df
    .filter(
        F.col("ranking") == 1
    )
    .select(
        "coluna",
        F.col("pct_estrutura").alias(
            "pct_predominancia"
        )
    )
    .withColumn(
        "pct_desvio_estrutura",
        100 - F.col("pct_predominancia")
    )
)

pattern_profile_df = (
    pattern_summary_df
    .filter(
        F.col("ranking") == 1
    )
    .select(
        "coluna",
        F.col("estrutura").alias(
            "estrutura_predominante"
        ),
        F.col("pct_estrutura").alias(
            "pct_predominancia"
        )
    )
    .join(
        pattern_concentration_df,
        "coluna",
        "left"
    )
    .join(
        pattern_diversity_df,
        "coluna",
        "left"
    )
    .join(
        pattern_regularity_df
        .select(
            "coluna",
            "classificacao_regularidade"
        ),
        "coluna",
        "left"
    )
    .join(
        pattern_deviation_summary_df
        .select(
            "coluna",
            "pct_desvio_estrutura"
        ),
        "coluna",
        "left"
    )
    .orderBy("coluna")
)

display(pattern_profile_df)

In [0]:
# ============================================================
# 05.14 RESUMO EXECUTIVO DO PATTERN PROFILE
# ============================================================

# O QUE FAZ:
# Cria uma visão executiva das principais características estruturais de cada coluna.

# COMO FAZ:
# Seleciona as métricas mais relevantes do perfil consolidado e cria uma interpretação automática.

# POR QUE É IMPORTANTE:
# Facilita a interpretação do Pattern Profile e fornece uma estrutura padronizada para o Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Qual é o nível de padronização estrutural de cada coluna?"

pattern_summary_executivo_df = (
    pattern_profile_df
    .withColumn(
        "leitura",
        F.when(
            F.col("pct_predominancia") >= 95,
            "Estrutura altamente padronizada."
        )
        .when(
            F.col("pct_predominancia") >= 80,
            "Estrutura predominantemente padronizada."
        )
        .when(
            F.col("pct_predominancia") >= 50,
            "Estrutura moderadamente padronizada."
        )
        .otherwise(
            "Alta variabilidade estrutural."
        )
    )
    .select(
        "coluna",
        "estrutura_predominante",
        "pct_predominancia",
        "estruturas_distintas",
        "concentracao_top1",
        "concentracao_top3",
        "concentracao_top5",
        "pct_desvio_estrutura",
        "classificacao_regularidade",
        "leitura"
    )
    .orderBy(
        F.desc("pct_predominancia")
    )
)

display(pattern_summary_executivo_df)

In [0]:
# ============================================================
# 05.15 VISUALIZAÇÃO — ESTRUTURAS PREDOMINANTES
# ============================================================

# O QUE FAZ:
# Prepara os dados para visualizar a estrutura predominante de cada coluna.

# COMO FAZ:
# Utiliza o Pattern Profile consolidado e seleciona a estrutura predominante e sua participação percentual.

# POR QUE É IMPORTANTE:
# Permite identificar visualmente quais colunas apresentam maior ou menor padronização.

# PERGUNTA RESPONDIDA:
# "Quais colunas possuem maior predominância estrutural?"

visualizacao_estrutura_df = (
    pattern_profile_df
    .select(
        "coluna",
        "estrutura_predominante",
        "pct_predominancia"
    )
    .orderBy(
        F.desc("pct_predominancia")
    )
)

display(visualizacao_estrutura_df)

In [0]:
# ============================================================
# 05.16 VISUALIZAÇÃO — COMPOSIÇÃO DE CARACTERES
# ============================================================

# O QUE FAZ:
# Prepara os dados para visualizar a composição dos valores nas colunas analisadas.

# COMO FAZ:
# Utiliza o percentual de cada categoria de composição.

# POR QUE É IMPORTANTE:
# Permite identificar rapidamente colunas com composição homogênea ou heterogênea.

# PERGUNTA RESPONDIDA:
# "Qual composição de caracteres predomina em cada coluna?"

visualizacao_composicao_df = (
    character_composition_df
    .select(
        "coluna",
        "composicao",
        "pct_composicao"
    )
    .orderBy(
        "coluna",
        F.desc("pct_composicao")
    )
)

display(visualizacao_composicao_df)

In [0]:
# ============================================================
# 05.17 VISUALIZAÇÃO — DESVIOS DE PADRÃO
# ============================================================

# O QUE FAZ:
# Prepara os dados para visualizar o percentual de registros que apresentam estrutura diferente do padrão predominante.

# COMO FAZ:
# Utiliza a métrica pct_desvio_estrutura calculada no perfil consolidado.

# POR QUE É IMPORTANTE:
# Permite identificar colunas que apresentam maior quantidade de valores estruturalmente diferentes do padrão dominante.

# PERGUNTA RESPONDIDA:
# "Quais colunas apresentam maior desvio estrutural?"

visualizacao_desvio_df = (
    pattern_profile_df
    .select(
        "coluna",
        "pct_predominancia",
        "pct_desvio_estrutura"
    )
    .orderBy(
        F.desc("pct_desvio_estrutura")
    )
)

display(visualizacao_desvio_df)